In [1]:
import sounddevice as sd
import numpy as np
import soundfile as sf
from src.audio_device import AudioDevice
from src.sharc import Sharc
import matplotlib.pyplot as plt
import librosa
import mido
import time
from src.measure_ir import *

print(sd.query_devices())

   0 Microsoft Sound Mapper - Input, MME (2 in, 0 out)
>  1 Analogue 1 + 2 (8- Focusrite US, MME (8 in, 0 out)
   2 Microphone Array (Intel® Smart , MME (4 in, 0 out)
   3 Microsoft Sound Mapper - Output, MME (0 in, 2 out)
<  4 Realtek HD Audio 2nd output (Re, MME (0 in, 2 out)
   5 Speakers (8- Focusrite USB Audi, MME (0 in, 8 out)
   6 Speakers (Realtek(R) Audio), MME (0 in, 2 out)
   7 Primary Sound Capture Driver, Windows DirectSound (2 in, 0 out)
   8 Analogue 1 + 2 (8- Focusrite USB Audio), Windows DirectSound (8 in, 0 out)
   9 Microphone Array (Intel® Smart Sound Technology (Intel® SST)), Windows DirectSound (4 in, 0 out)
  10 Primary Sound Driver, Windows DirectSound (0 in, 2 out)
  11 Realtek HD Audio 2nd output (Realtek(R) Audio), Windows DirectSound (0 in, 2 out)
  12 Speakers (8- Focusrite USB Audio), Windows DirectSound (0 in, 8 out)
  13 Speakers (Realtek(R) Audio), Windows DirectSound (0 in, 2 out)
  14 Speakers (8- Focusrite USB Audio), Windows WASAPI (0 in, 2 out)
  1

In [2]:
arthur, sr = sf.read('data/arthur_clip_48k.wav')

fs = 96_000
arthur_96 = librosa.resample(arthur, orig_sr=sr, target_sr=fs)


ad = AudioDevice(fs, in_device=1, out_device=4)

In [3]:
sharc = Sharc(arthur_96, ad)


Using output: USB2.0-MIDI 1
Using input:  USB2.0-MIDI 0


In [10]:
sharc.no_cancel(1)

In [11]:
def string_header(sharc_freq, source_freq, sharc_amp, source_amp):
    sharc_freq = 0 if sharc_freq is None else sharc_freq
    source_freq = 0 if source_freq is None else source_freq

    return f'pfreq-{sharc_freq}_sfreq-{source_freq}_pamp-{sharc_amp}_samp-{source_amp}'



def play_tones(sharc_freq=None, source_freq=None, sharc_amp=0.5, source_amp=1.0):
    if sharc_freq is not None:
        sharc.pure_tone(sharc_freq, sharc_amp)

    silent = np.zeros(2 * fs, dtype=np.float32)
    if source_freq is not None:
        t = np.arange(2 * fs, dtype=np.float32) / fs
        source_tone = source_amp * np.sin(2.0 * np.pi * source_freq * t)
    else:
        source_tone = silent

    try:
        error, ref = ad.play(left=source_tone, right=silent)
    finally:
        if sharc_freq is not None:
            sharc.stop_tone()

    return error[fs:], ref[fs:], string_header(sharc_freq, source_freq, sharc_amp, source_amp)

In [12]:
panel_freqs = [250, 275, 300, 325, 350, 375, 400, 425, 450, 475, 500]
source_freqs = [700, 800, 900, 1000, 1100, 1200, 1300, 1400, 1500]
panel_amps = [0.15, 0.25, 0.4, 0.5]

In [13]:
n_samples = fs

error_panel = np.empty((len(panel_freqs), len(panel_amps), n_samples), dtype=np.float32)
ref_panel = np.empty_like(error_panel)

error_source = np.empty((len(source_freqs), n_samples), dtype=np.float32)
ref_source = np.empty_like(error_source)

error_both = np.empty(
    (len(panel_freqs), len(source_freqs), len(panel_amps), n_samples),
    dtype=np.float32
)
ref_both = np.empty_like(error_both)

# Panel-only recordings
for i, pf in enumerate(panel_freqs):
    for k, pa in enumerate(panel_amps):
        e, r, _ = play_tones(
            sharc_freq=pf,
            source_freq=None,
            sharc_amp=pa
        )
        error_panel[i, k] = e
        ref_panel[i, k] = r

# Source-only recordings
for j, sf in enumerate(source_freqs):
    e, r, _ = play_tones(
        sharc_freq=None,
        source_freq=sf
    )
    error_source[j] = e
    ref_source[j] = r

# Both-on recordings
for i, pf in enumerate(panel_freqs):
    for j, sf in enumerate(source_freqs):
        for k, pa in enumerate(panel_amps):
            e, r, _ = play_tones(
                sharc_freq=pf,
                source_freq=sf,
                sharc_amp=pa
            )
            error_both[i, j, k] = e
            ref_both[i, j, k] = r


In [14]:
filename = 'distortion_side.npz'
panel_to_err_cm = 46
ir_len = 256

In [15]:
irs = np.load('./data/current_irs.npz')
error_ir_raw = irs['error_ir']
ref_ir_raw = irs['ref_ir']
error_ir, ref_ir = align_irs_by_distance(error_ir_raw, ref_ir_raw, distance_cm=panel_to_err_cm, fs=fs, ir_len=ir_len)

np.savez(
    f'./data/distortion/{filename}',

    error_ir=error_ir,
    ref_ir=ref_ir,
    error_ir_raw=error_ir_raw,
    ref_ir_raw=ref_ir_raw,

    fs=fs,
    ir_len=ir_len,
    panel_to_err_cm=panel_to_err_cm,


    panel_freqs=panel_freqs,
    source_freqs=source_freqs,
    panel_amps=panel_amps,
    source_amp=1.0,

    error_panel=error_panel,
    ref_panel=ref_panel,
    error_source=error_source,
    ref_source=ref_source,
    error_both=error_both,
    ref_both=ref_both,

)